# Data Persistence



## from events to durable data

- An ingestion tool is responsible to collect events from a source
- A streaming tool is responsible to make the events available in real-time
- A processing tool is responsible to process the events transforming/enriching/aggregating them

:::{.fragment .callout-caution}
All these tools are designed to handle events in memory or for limited time periods, but what happens when the events need to be stored for later use? This is where data persistence comes into play.
:::


## A brief history of data persistence

- **1956 — Direct-access storage:** IBM RAMAC brought random-access disk storage to business computing, making stored data much easier to retrieve and update than punched-card workflows. ([IBM](https://www.ibm.com/history/ramac))

- **1970 — Relational databases:** Codd proposed organizing data as relations, reducing the need for users to know its physical layout. ([Codd, 1970](https://doi.org/10.1145/362384.362685))

- **1987 — Persistence as a research topic:** Atkinson and Buneman surveyed how programming languages could integrate with databases and support persistent data. ([ACM survey](https://doi.org/10.1145/62070.45066))

- **2003 — Distributed files:** Google File System was designed to store and serve large data sets across clusters of commodity machines. ([Google Research](https://research.google/pubs/the-google-file-system/))

- **Today — Managed tables over files:** Table formats such as Apache Iceberg track table metadata and snapshots across data files, adding a table abstraction over distributed storage. ([Apache Iceberg](https://iceberg.apache.org/docs/latest/))

:::{.fragment .callout-note}
The focus has shifted from keeping data after a program ends to managing large, distributed collections of files as usable tables.
:::

## Persistence layers

| Layer | Answers the question | Examples |
|---|---|---|
| **File format** | How are records and values encoded in a file? | Parquet, Avro, ORC, JSON, CSV |
| **Table format** | Which files make up a table, and what is its current schema/state? | Apache Iceberg, Delta Lake, Apache Hudi |
| **Storage** | Where do the bytes live, and how can workers access them? | HDFS, S3 / S3-compatible (RustFS, MinIO), cloud object storage, local disk |

:::{.fragment .callout-note}
These layers can be seen as a pyramid, where each layer builds on the one below it. Storage is the foundation; file formats encode the bytes stored there; a table format organizes many files into one logical table.
:::

# File formats

## What is a file format?

A file format defines the physical encoding of records and values. CSV and JSON are easy to inspect, but are text formats with limited type information and higher parsing cost.

Analytical pipelines commonly use **columnar** formats (Parquet, ORC) for scans and **row-oriented** formats (Avro) for record-at-a-time exchange. The choice depends on the workload; a format by itself does not provide a database table or a storage service.

## Let's compare file formats using a real file example

::::{.columns}
:::{.column width="50%"}
A 50,000-row sample from the [NYC 311 Service Requests open dataset](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9) 
The sample covers January 2024, files are encoded and analyzed using a python script `compare_file_formats.py` 
:::

:::{.column width="50%"}

```bash
cd tap2027
docker build --tag tap:python ./python
mkdir -p python/data
docker run --rm \
  -v "$(pwd)/python/bin:/usr/src/app/bin" \
  -v "$(pwd)/python/data:/usr/src/app/data" \
  -e PYTHON_APP=compare_file_formats.py tap:python
```
:::
:::: 

## CSV baseline

::::{.columns}
:::{.column width="50%"}
- `unique_key`: `60191146`
- `created_date`: `2024-01-31T23:59:55.000`
- `agency`: `NYPD`
- `complaint_type`: `Illegal Parking`
- `borough`: `BROOKLYN`
- `latitude`: `40.62193209452144`
- `longitude`: `-74.0009473874934`

:::
:::{.column width="50%" .fragment}
The 50,000-row CSV baseline is **5,636,715 bytes (5.64 MB)**.
:::
::::

## Parquet: columnar analytics

<https://parquet.apache.org/>

::::{.columns}
:::{.column width="50%"}
> Apache Parquet is an open source, column-oriented data file format designed for efficient data storage and retrieval. It provides high performance compression and encoding schemes to handle complex data in bulk and is supported in many programming languages and analytics tools.

Parquet stores data by column within row groups. For the NYC 311 data, a query that needs only `borough` and `complaint_type` can avoid reading unrelated columns such as `latitude` and `longitude`. Encodings, compression, and per-column statistics can reduce I/O and help engines skip irrelevant row groups.

:::
:::{.column width="50%" .fragment}

```text
row group 0: borough block | complaint_type block | latitude block | ...
row group 1: borough block | complaint_type block | latitude block | ...
```

Parquet is a strong default for analytical data files. 

Uncompressed Parquet of the NYC 311 data is **2,630,699 bytes (2.63 MB)** for 50,000 rows. 
:::
::::

## DuckDB: the swiss army knife of analytics
<https://duckdb.org/>

DuckDB is an in-process analytical database management system that allows for the execution of SQL queries on data stored in various formats, including CSV, Parquet, and more. It is designed to be fast and efficient for analytical workloads.

:::{.fragment}
```
duckdb 

D select * from 'nyc311.parquet' limit 5;
┌────────────┬──────────────────────┬─────────┬───┬──────────┬────────────────────┬────────────────────┐
│ unique_key │     created_date     │ agency  │ … │ borough  │      latitude      │     longitude      │
│  varchar   │       varchar        │ varchar │   │ varchar  │       double       │       double       │
├────────────┼──────────────────────┼─────────┼───┼──────────┼────────────────────┼────────────────────┤
│ 60191146   │ 2024-01-31T23:59:5…  │ NYPD    │ … │ BROOKLYN │  40.62193209452144 │  -74.0009473874934 │
│ 60194494   │ 2024-01-31T23:59:1…  │ HPD     │ … │ BRONX    │  40.85455863917999 │ -73.90015830400407 │
│ 60192234   │ 2024-01-31T23:59:0…  │ DOT     │ … │ QUEENS   │                    │                    │
│ 60200052   │ 2024-01-31T23:59:0…  │ TLC     │ … │ QUEENS   │  40.64832048620134 │ -73.78828125130184 │
│ 60199465   │ 2024-01-31T23:59:0…  │ NYPD    │ … │ BROOKLYN │ 40.699800319891985 │ -73.98361593748928 │
├────────────┴──────────────────────┴─────────┴───┴──────────┴────────────────────┴────────────────────┤
│ 5 rows                                                                           7 columns (6 shown) │
└──────────────────────────────────────────────────────────────────────────────────────────────────────┘
```
:::


## ORC: columnar analytics

<https://orc.apache.org/>

::::{.columns}
:::{.column width="50%"}
ORC (Optimized Row Columnar) stores data in **stripes**, grouping values by column. Indexes and statistics can help query engines skip stripes that cannot match a filter.

For NYC 311 data, a query that needs only `borough` and `complaint_type` can avoid reading unrelated columns such as `latitude` and `longitude`.
:::
:::{.column width="50%" .fragment}
```text
stripe 0: borough [..] | complaint_type [..] | latitude [..] | ...
stripe 1: borough [..] | complaint_type [..] | latitude [..] | ...
```

Uncompressed ORC is **3,780,089 bytes (3.78 MB)** for 50,000 rows. The checked borough counts and latitude statistics match the CSV baseline.
:::
::::

### Example: reading ORC with PyArrow

In [1]:
import duckdb
import pyarrow.orc as orc

table = orc.read_table("../python/data/formats-demo/nyc311.orc")
con = duckdb.connect()
con.register("requests", table)

con.sql("""
    SELECT borough, COUNT(*) AS incidents
    FROM requests
    GROUP BY borough
    ORDER BY incidents DESC
""").show()

ModuleNotFoundError: No module named 'duckdb'

## Avro: row-oriented serialization
<https://avro.apache.org/>

> A data serialization system

Avro stores records in a compact binary form and uses an explicit schema. Its object-container files store the writer schema with the data, making Avro useful for record exchange and event/streaming pipelines. ([Apache Avro documentation](https://avro.apache.org/docs/1.12.0/))

A small Avro record schema looks like this:

```json
{
  "type": "record",
  "name": "ServiceRequest",
  "fields": [
    {"name": "agency", "type": "string"},
    {"name": "borough", "type": ["null", "string"], "default": null}
  ]
}
```


Uncompressed Avro is **4,211,819 bytes (4.21 MB)** for 50,000 rows.

### Example: reading Avro with fastavro

In [1]:
from fastavro import reader

with open("../python/data/formats-demo/nyc311.avro", "rb") as f:
    avro = reader(f)
    print(avro.writer_schema)
    for i, record in enumerate(avro):
        print(record)
        if i == 4:
            break

{'type': 'record', 'name': 'NYC311Request', 'fields': [{'name': 'unique_key', 'type': 'string'}, {'default': None, 'name': 'created_date', 'type': ['null', 'string']}, {'default': None, 'name': 'agency', 'type': ['null', 'string']}, {'default': None, 'name': 'complaint_type', 'type': ['null', 'string']}, {'default': None, 'name': 'borough', 'type': ['null', 'string']}, {'default': None, 'name': 'latitude', 'type': ['null', 'double']}, {'default': None, 'name': 'longitude', 'type': ['null', 'double']}]}
{'unique_key': '60191146', 'created_date': '2024-01-31T23:59:55.000', 'agency': 'NYPD', 'complaint_type': 'Illegal Parking', 'borough': 'BROOKLYN', 'latitude': 40.62193209452144, 'longitude': -74.0009473874934}
{'unique_key': '60194494', 'created_date': '2024-01-31T23:59:13.000', 'agency': 'HPD', 'complaint_type': 'HEAT/HOT WATER', 'borough': 'BRONX', 'latitude': 40.85455863917999, 'longitude': -73.90015830400407}
{'unique_key': '60192234', 'created_date': '2024-01-31T23:59:08.000', 'age

## Arrow IPC: columnar exchange

<https://arrow.apache.org/docs/python/ipc.html>

::::{.columns}
:::{.column width="50%"}
Arrow IPC serializes Arrow **record batches** to a file or stream. Each batch stores columns in contiguous buffers for efficient exchange between Arrow-aware tools. Unlike ORC, IPC is an interchange format, not a table format with indexes for pruning data.

For NYC 311 data, downstream code can select the fields it needs from the in-memory table.
:::
:::{.column width="50%" .fragment}
```text
IPC file: schema + record batches
batch 0: borough [..] | complaint_type [..] | latitude [..] | ...
batch 1: borough [..] | complaint_type [..] | latitude [..] | ...
```

The uncompressed IPC file is **4,687,954 bytes (4.69 MB)** for 50,000 rows. Its checked borough counts and latitude statistics match the CSV baseline.
:::
::::

### Example: reading IPC with PyArrow

In [2]:
import pyarrow as pa

with pa.memory_map("../python/data/formats-demo/nyc311.arrow", "r") as source:
    table = pa.ipc.open_file(source).read_all()

print(table.select(["borough", "complaint_type", "latitude"]).slice(0, 5))

ModuleNotFoundError: No module named 'pyarrow'

## File formats: choose by workload

| Format | Organization | Useful for | Trade-off |
|---|---|---|---|
| CSV | Text, rows | Simple exchange and inspection | Types and nested structures need conventions |
| JSON | Text, records | Readable nested events and APIs | Parsing and repeated field names can be expensive |
| Avro | Binary, rows with schemas | Record exchange, schema-aware events | Not optimized for reading a few columns |
| Parquet / ORC | Binary, columns | Compressed analytical scans | Not human-readable; less suited to single-record edits |
| Arrow IPC | Columnar, file or stream | Fast interchange with Arrow-aware tools | Not a table format; ecosystem and compression differ |

Compression reduces bytes, not necessarily execution time; measure with real data. The same dataset may appear in different formats at different pipeline stages.

## Final comparison: file sizes on the NYC 311 sample

| Format | File size (bytes) | File size (MB) |
|---|---:|---:|
| CSV | 5,636,715 | 5.64 |
| Avro | 4,211,819 | 4.21 |
| Parquet | 2,630,699 | 2.63 |
| ORC | 3,780,089 | 3.78 |
| Arrow IPC | 4,687,954 | 4.69 |

All formats preserved 50,000 rows and the same checked borough and latitude summaries. Parquet was the smallest in this uncompressed run; this is a file-size comparison, not a universal performance ranking.

# Open table formats

## Why and when did they emerge?

In the late 2010s, data lakes increasingly held many Parquet files on HDFS and object stores. Teams wanted analytics, concurrent ingestion, updates, reliable reads, and multiple engines over the same data.

A plain file directory and traditional partition listings made it difficult to coordinate atomic changes and track exactly which files formed a consistent table. New table formats emerged: **Apache Hudi** grew from Uber's Hoodie work, **Apache Iceberg** from Netflix, and **Delta Lake** from Databricks. They are open projects with distinct metadata protocols.

They build on earlier database and transactional-file-system ideas; this is an evolution, not the invention of transactions.

## Files alone are not a table

A directory of Parquet files does not inherently say which files are current, how a schema changed, or which set of files represented the table at a previous point in time.

A query engine can read files, but concurrent writers, deletes, schema changes, and consistent snapshots require a shared table-level protocol and metadata.

## Table format vs database

| | Open table format | Database management system |
|---|---|---|
| Main role | Specify table metadata, file membership and commit rules | Run a system that stores, queries and manages data |
| Execution | Requires a compatible engine, catalog and storage | Typically includes query execution and storage management |
| Typical strengths | Shared analytical data and interoperability between engines | Depending on the DBMS: constraints, indexes, fine-grained access, low-latency transactions |
| Examples | Iceberg, Delta Lake | PostgreSQL, DuckDB, ClickHouse |

A format is not a running DBMS; a DBMS can also read or write an open table format. Capabilities and guarantees depend on the specific engine, catalog, and supported operations.

## Apache Iceberg: snapshots over data files

<https://iceberg.apache.org/docs/latest/>

::::{.columns}
:::{.column width="50%"}
Iceberg tracks a table's data files, schema, partition specification, and snapshots in metadata. Engines use a catalog to find the current metadata and commit new table states.

It is designed for large analytic tables across engines, with schema and partition evolution and snapshot-based reads. The catalog, storage, and engine must be configured to work together.
:::
:::{.column width="50%" .fragment}
```text
catalog -> current metadata
             snapshot -> manifests -> data files
             snapshot -> manifests -> data files
```
:::
::::

### Iceberg example: NYC 311 to a table

The latest available compatible pairing is the official `apache/spark:4.1.4` image with Iceberg `1.12.0` (`iceberg-spark-runtime-4.1_2.13`). 

After generating `python/data/formats-demo/nyc311.parquet`, run this from the `tap2027` project root. The code is mounted read-only; the app writes its Iceberg warehouse to `python/data/formats-demo/iceberg`.

```bash
docker run --rm \
  --user 0:0 \
  -v "$PWD/spark/code:/opt/spark/work-dir/code:ro" \
  -v "$PWD/python/data/formats-demo:/opt/spark/work-dir/input" \
  apache/spark:4.1.4 \
  /opt/spark/bin/spark-submit \
  --packages org.apache.iceberg:iceberg-spark-runtime-4.1_2.13:1.12.0 \
  /opt/spark/work-dir/code/nyc311_iceberg.py
```

## Delta Lake: a transaction log for tables

<https://docs.delta.io/>

::::{.columns}
:::{.column width="50%"}
Delta Lake records table versions in the `_delta_log` alongside Parquet data files. A commit adds a new version, allowing supported engines to read a consistent snapshot and perform operations such as updates, deletes, and schema evolution.

Reader and writer support depends on the Delta protocol features used by the table.
:::
:::{.column width="50%" .fragment}
```text
_delta_log/
  00000000000000000000.json
  00000000000000000001.json
part-....parquet
part-....parquet
```
:::
::::

### Delta example: NYC 311 from Parquet

The source stays `nyc311.parquet`; Delta writes Parquet data files plus `_delta_log/` under `python/data/formats-demo/delta/nyc311`.

Delta Lake `4.4.1` provides a Spark 4.1 / Scala 2.13 artifact. Reuse the official Spark `4.1.4` image already used for Iceberg, so no additional Spark image is needed. After generating the Parquet sample, run this from the `tap2027` project root:

```bash
docker run --rm \
  --user 0:0 \
  -v "$PWD/spark/code:/opt/spark/work-dir/code:ro" \
  -v "$PWD/python/data/formats-demo:/opt/spark/work-dir/input" \
  apache/spark:4.1.4 \
  /opt/spark/bin/spark-submit \
  --packages io.delta:delta-spark_4.1_2.13:4.4.1 \
  /opt/spark/work-dir/code/nyc311_delta.py
```

The code is mounted read-only. This workspace's data directory is root-owned, so the container runs as root to write the table there.

## Apache Hudi: tables for changing data

<https://hudi.apache.org/docs/overview/>

::::{.columns}
:::{.column width="50%"}
Hudi tracks commits on a table timeline and supports inserts, updates, deletes, and incremental reads. This can suit ingestion pipelines where records are repeatedly updated, as well as analytical queries.

Copy-on-Write organizes updates into new base files; Merge-on-Read combines base files with logs and compacts them later. Engine support varies by operation and table type.
:::
:::{.column width="50%" .fragment}
```text
timeline: commit 1 -> commit 2 -> commit 3
file group:
  base file (Parquet) + log files
```
:::
::::

### Hudi example: NYC 311 from Parquet

This demo uses a Copy-on-Write table: Parquet base files plus Hudi's `.hoodie/` timeline. `unique_key` identifies records and `created_date` orders changes.

Run this from the `tap2027` project root:

```bash
docker run --rm \
  --user 0:0 \
  -v "$PWD/spark/code:/opt/spark/work-dir/code:ro" \
  -v "$PWD/python/data/formats-demo:/opt/spark/work-dir/input" \
  apache/spark:4.1.4 \
  /opt/spark/bin/spark-submit \
  --packages org.apache.hudi:hudi-spark4.1-bundle_2.13:1.2.1 \
  /opt/spark/work-dir/code/nyc311_hudi.py
```


## NYC 311: table format comparison

Same simple benchmark `GROUP BY borough, COUNT(*)` was run on the three table formats, reading 50,000 rows from each. The table sizes and query times are shown below.

| Input / table | Files | Parquet data | Other files* | Total on disk | Query median |
|---|---:|---:|---:|---:|---:|
| Parquet input | 1 | 2.631 MB | 0.000 MB | 2.631 MB | 74.7 ms |
| Iceberg | 10 | 0.991 MB | 0.022 MB | 1.013 MB | 73.2 ms |
| Delta | 6 | 1.251 MB | 0.015 MB | 1.266 MB | 57.3 ms |
| Hudi | 52 | 2.182 MB | 0.120 MB | 2.302 MB | 117.1 ms |

:::{.fragment .callout-note}
This is a naive comparison, just to illustrate the table formats. A reliable performance comparison needs larger data and controlled file layout, compression, cache state, and Spark resources.
:::


## What does ACID mean here?

- **Atomicity:** a supported table commit is visible as a whole, not as half of a set of new files.
- **Consistency:** committed changes must respect the table format's rules; business invariants and constraints still need explicit enforcement.
- **Isolation:** readers see a committed snapshot while another writer commits; conflict detection and isolation depend on operation and implementation.
- **Durability:** committed metadata and files survive only as reliably as the catalog and underlying storage; plan backups and retention.


# Shared and distributed storage

## Storage

Storage determines durability, capacity, access patterns, failure behavior, and how workers find the same data.

- **Local disk**: fast and useful for scratch data, but normally tied to one machine.
- **SAN/Virtual Disk**: block-level storage accessed over a network; often used for databases and virtual machines.
- **HDFS**: distributed filesystem; files are split into blocks managed by DataNodes, with namespace/metadata managed by a NameNode.
- **Object storage**: objects in buckets, accessed through an API; examples include Amazon S3, MinIO and RustFS (S3-compatible, self-hosted), Google Cloud Storage, and Azure Blob Storage.

## HDFS recap: shared namespace for a cluster

Recall the HDFS roles from Flume: the NameNode manages filesystem namespace and block locations; DataNodes store blocks. Files are split into blocks and can be replicated across DataNodes for fault tolerance.

From a client or Spark worker, a path such as `hdfs://namenode:9000/warehouse/...` names data in the shared filesystem. A worker does not need the file to exist on its own local disk.

HDFS is a distributed filesystem, not an object store and not a table format.

## Object storage: self-hosted and cloud services

Object storage keeps **objects in buckets** and exposes them through an HTTP API, most commonly the **Amazon S3 API**. Public-cloud services include Amazon S3, Google Cloud Storage, and Azure Blob Storage; S3-compatible software such as MinIO, RustFS, or Ceph can be run on premises.

Object stores are the most common storage layer for lakehouses: they are durable, scalable, relatively inexpensive, and accessible to many engines simultaneously.

## Object storage: the model

::::{.columns}
:::{.column width="50%"}
- **Bucket:** a top-level container with its own policies, versioning, and lifecycle rules.
- **Object:** bytes + metadata, addressed by a **key** such as `raw/nyc311/2024-01.parquet`.
- **Flat namespace:** `/` is just a character in the key; "folders" are prefixes shown by clients.
- **HTTP API:** `PUT`, `GET` (including byte ranges), `LIST`, `DELETE`, multipart upload for large objects.
- **Whole-object writes:** objects are replaced, not edited in place.
:::
:::{.column width="50%" .fragment}
```text
s3://lake/                       <- bucket
  raw/nyc311/2024-01.parquet     <- object key
  warehouse/course/nyc311/
    data/00000-...parquet
    metadata/v3.metadata.json
```

Range `GET`s let Parquet readers fetch only the footer and the column chunks they need.
:::
::::

## Filesystem, HDFS, or object store?

| | Local / POSIX filesystem | HDFS | Object storage (S3 API) |
|---|---|---|---|
| Namespace | Hierarchical directories | Hierarchical, kept by the NameNode | Flat keys in buckets |
| Access | System calls, byte-level edits | HDFS client, append-only files | HTTP API, whole-object writes |
| Rename | Atomic | Atomic (metadata operation) | Usually **copy + delete**, not atomic |
| Scaling | One machine | Cluster; NameNode memory limits file count | Scales horizontally; can be pay-per-use |
| Compute coupling | Same machine | Usually co-located with compute | Separate from compute |

:::{.fragment .callout-important}
Object stores lack atomic rename and cheap directory listing. That is a main reason why table formats track files in metadata and commit through a catalog or conditional write, rather than renaming directories.
:::

## MinIO: the reference self-hosted S3 store

<https://github.com/minio/minio>

::::{.columns}
:::{.column width="50%"}
- Created by **MinIO, Inc.** (founded 2014) as a high-performance, S3-compatible object store written in Go.
- A single binary or container image: easy to run on a laptop, on bare metal, or on Kubernetes (operator).
- It became a common on-premises S3 and lakehouse storage target in teaching labs, CI pipelines, and private clouds.
:::
:::{.column width="50%" .fragment}
**Design ideas**

- **Erasure coding:** each object is split into data and parity shards across drives or nodes; it survives the loss of up to the parity count.
- **Bitrot protection:** shards are checksummed and verified when read.
- **No separate metadata database:** object metadata is stored next to the data (`xl.meta`).
- **Server pools:** the cluster expands by adding pools.
:::
::::

## MinIO features relevant to a data pipeline

| Feature | What it does | Pipeline use |
|---|---|---|
| **Versioning** | Keeps previous versions of an overwritten or deleted object | Recover from bad writes |
| **Object lock (WORM)** | Prevents deletion or overwrite for a retention period | Compliance and audit data |
| **Lifecycle (ILM)** | Expires or moves objects by age or prefix | Clean temporary files and old raw data |
| **Bucket notifications** | Publishes events for object create/delete to Kafka, webhooks, AMQP, and more | Trigger ingestion when a file lands |
| **Replication** | Copies buckets to another site or cluster | Disaster recovery |
| **IAM policies, TLS, KMS encryption** | Controls access and encrypts data | Least privilege for Spark, ClickHouse, and the assistant |

:::{.fragment .callout-tip}
Bucket notifications to Kafka connect object storage to the streaming part of the course: *a file arrives -> an event is published -> a consumer processes it*.
:::

## MinIO in 2026: licensing and lifecycle

::::{.columns}
:::{.column width="55%"}
- **2021:** the server changed licence from Apache 2.0 to **AGPLv3**.
- **2025:** administration features were removed from the community web console; the community edition then became **source-only** (no official binaries or images).
- **3 Dec 2025:** the community repository entered **maintenance mode**.
- **25 Apr 2026:** the GitHub repository was **archived** and became read-only.
- The vendor's current products are **AIStor** (Free and Enterprise editions).
:::
:::{.column width="45%" .fragment}
**What this means for us**

- The **S3 API concepts** and client configuration still apply to any S3-compatible store.
- Official community images are no longer available from Docker Hub or Quay, and the code receives **no security fixes**.
- The licence is not the blocker for a lab (AGPLv3 obligations apply when you modify it and offer it over a network); **lack of maintenance** is.
- Alternatives: **RustFS** (our choice), **Garage**, **SeaweedFS**, **Ceph RGW**, or a cloud object store.
:::
::::

:::{.fragment .callout-caution}
The licence, governance, and maintenance status of an open-source component are architectural risks. Keep storage behind the standard S3 API so it can be replaced.
:::

## RustFS: our choice for the course

<https://github.com/rustfs/rustfs> · <https://rustfs.com>

::::{.columns}
:::{.column width="50%"}
- S3-compatible distributed object storage written in **Rust**, under the **Apache 2.0** licence.
- Designed as a MinIO replacement: same deployment model (single binary or container, `/data` volumes), erasure coding, web console, and an `mc`-style CLI called **`rc`**.
- Official multi-architecture images on Docker Hub (`rustfs/rustfs`); **1.0** was released in September 2026, and we use **1.0.5**.
:::
:::{.column width="50%" .fragment}
**Why we use it**

- Actively maintained, with a permissive licence and official images: no source build is needed.
- Spark, the Iceberg REST catalog, and DuckDB connect to it through the **same S3 settings** as MinIO; only the endpoint changes.

**Caveats**

- It is a young project: check S3 feature coverage (versioning, object lock, notifications) before relying on it in production.
- A single-drive lab setup has **no parity** (RustFS logs a warning); use multiple drives or nodes for redundancy.
:::
::::

## MinIO vs RustFS at a glance

| | MinIO (community) | RustFS |
|---|---|---|
| Language | Go | Rust |
| Licence | AGPLv3 (Apache 2.0 before 2021) | Apache 2.0 |
| Status (Nov 2026) | Archived; vendor focuses on AIStor | Active; 1.0.x releases |
| Distribution | Source only | Official Docker images and binaries |
| API | Amazon S3 | Amazon S3 (path-style by default) |
| CLI | `mc` | `rc` (`mc`-style commands) |
| Console | `:9001` | `:9001`, at `/rustfs/console/` |
| Durability | Erasure coding, bitrot protection | Erasure coding |

:::{.fragment .callout-tip}
Because applications use only the S3 API, we swapped storage by changing **one image and one endpoint**. Spark, Iceberg, and the catalog code stayed the same.
:::

### RustFS hands-on: start a server and create a bucket

```bash
cd quickstart/pizza-lakehouse
docker compose up -d --wait rustfs
```

- `127.0.0.1:19000` is the S3 API endpoint. The web console is at <http://localhost:19001/rustfs/console/index.html> (user `tapadmin`, password `tapsecret123`).
- These root credentials are for the lab only. Create a dedicated user and policy for each application.

Create a bucket and upload the NYC 311 Parquet sample generated earlier:

```bash
docker compose cp ../../python/data/formats-demo/nyc311.parquet rustfs:/tmp/nyc311.parquet
docker compose exec rustfs sh -c '
  rc alias set local http://localhost:9000 tapadmin tapsecret123 &&
  rc bucket create --ignore-existing local/lake &&
  rc put /tmp/nyc311.parquet local/lake/raw/nyc311/nyc311.parquet &&
  rc object list --recursive local/lake'
```

### Reading object storage from DuckDB and Spark

The `nyc311.parquet` object uploaded in the previous step can be queried in place, without downloading it first.

::::{.columns}
:::{.column width="50%"}
**DuckDB** (`httpfs` extension, from the host)

```sql
INSTALL httpfs; LOAD httpfs;
CREATE SECRET lab (
  TYPE s3, KEY_ID 'tapadmin', SECRET 'tapsecret123',
  ENDPOINT 'localhost:19000', URL_STYLE 'path',
  USE_SSL false, REGION 'us-east-1');

SELECT borough, count(*) AS incidents
FROM 's3://lake/raw/nyc311/nyc311.parquet'
GROUP BY borough ORDER BY incidents DESC;
```
:::
:::{.column width="50%" .fragment}
**Spark** (`s3a://` connector, inside the demo network)

```bash
docker run --rm --network pizza-lakehouse_default \
  apache/spark:4.1.4 /opt/spark/bin/spark-sql -S \
  --conf spark.driver.extraJavaOptions=-Divy.home=/tmp/ivy \
  --packages org.apache.hadoop:hadoop-aws:3.4.2 \
  --conf spark.hadoop.fs.s3a.endpoint=http://rustfs:9000 \
  --conf spark.hadoop.fs.s3a.endpoint.region=us-east-1 \
  --conf spark.hadoop.fs.s3a.path.style.access=true \
  --conf spark.hadoop.fs.s3a.connection.ssl.enabled=false \
  --conf spark.hadoop.fs.s3a.access.key=tapadmin \
  --conf spark.hadoop.fs.s3a.secret.key=tapsecret123 \
  -e 'SELECT borough, count(*) FROM parquet.`s3a://lake/raw/nyc311/` GROUP BY borough'
```

`hadoop-aws` must match Spark's Hadoop version (3.4.2 in Spark 4.1.4).
:::
::::

:::{.fragment .callout-note}
Self-hosted stores usually need **path-style** URLs (`http://host:9000/bucket/key`) rather than virtual-host URLs (`http://bucket.host/key`). The Iceberg demo does not use `s3a`: Iceberg's own `S3FileIO` reads and writes the table files.
:::

## Why table formats and object storage fit together

| Object-store limitation | How a table format handles it |
|---|---|
| Listing millions of keys is slow and costly | Metadata and manifests list the data files, so engines do not need to list directories |
| No atomic rename | A commit atomically updates **one pointer**: through a catalog (Iceberg) or a log entry written with a conditional put/log store (Delta) |
| Objects are immutable | Changes write new files and a new snapshot; old files remain for time travel until expired |
| Many small files hurt performance | Compaction rewrites small files into larger ones as a maintenance job |

:::{.fragment .callout-warning}
Our local Iceberg demo uses a `hadoop` catalog on the local filesystem. On object storage, its commit relies on rename and is **not safe for concurrent writers**; use a REST, JDBC, Hive, or Glue catalog instead.
:::

# Lakehouse

## From warehouses to lakes (1988–2015)

| Period | Milestone | What it added / what was missing |
|---|---|---|
| **1988–1996** | **Data warehouse**: Devlin & Murphy (IBM, 1988), Inmon (1992), Kimball (1996) | Integrated, structured, SQL, ACID — but expensive storage that is coupled to compute and works poorly with semi-structured data |
| **2006–2010** | **Hadoop** and the **data lake** (term coined by James Dixon, 2010) | Cheap storage of any format, schema-on-read — but no transactions, weak governance: the "data swamp" |
| **2011–2015** | **Cloud warehouses**: BigQuery, Redshift, Snowflake | Elastic compute separated from storage — but data is copied from the lake into proprietary storage |

:::{.fragment .callout-note}
The **two-tier** architecture (lake for raw data and ML + warehouse for BI) required ETL copies, causing staleness, extra cost, and two sets of permissions. The lakehouse aims to remove the second copy.
:::

## From lakes to lakehouses (2016–2026)

| Period | Milestone | What it added / what was missing |
|---|---|---|
| **2016–2019** | **Open table formats**: Hudi (Uber), Iceberg (Netflix), Delta Lake (Databricks) | ACID tables directly on lake files |
| **2020–2021** | **"Lakehouse"**: Databricks blog (2020) and CIDR paper (Armbrust et al., 2021) | Names the architecture: one copy of the data, warehouse features on open storage |
| **2023–2026** | **Convergence**: interoperability (Delta UniForm, Apache XTable), open catalogs (Unity Catalog OSS, Apache Polaris), AWS S3 Tables, DuckLake, Iceberg spec v3 | Competition shifts from file formats to **catalogs** and governance |

## What is a lakehouse?

> A data management system based on low-cost and directly-accessible storage that also provides traditional analytical DBMS management and performance features such as ACID transactions, data versioning, auditing, indexing, caching, and query optimization.
>
> — Armbrust, Ghodsi, Xin, Zaharia, [*Lakehouse*, CIDR 2021](https://www.cidrdb.org/cidr2021/papers/cidr2021_paper17.pdf)

::: {.fragment}
**Core properties**

1. **Open file formats** (Parquet, ORC, Avro) on **object storage** or HDFS.
2. A **table format** that provides ACID commits, snapshots, and schema/partition evolution.
3. A **catalog** that names tables, coordinates commits, and enforces access control.
4. **Multiple engines** (Spark, Trino, Flink, DuckDB, ClickHouse) on the **same copy** of the data.
5. Both **BI/SQL** and **data science/ML** workloads, batch and streaming.
:::

## Warehouse, lake, and lakehouse

| | Data warehouse | Data lake | Lakehouse |
|---|---|---|---|
| Data | Structured, curated | Any format, often raw | Any format; curated tables on top |
| Schema | On write | On read | On write for tables, evolvable |
| Transactions | Yes | No | Yes, through the table format |
| Storage | Managed by the DBMS, often proprietary | Open files on cheap storage | Open files on cheap storage |
| Engines | Usually the warehouse's own engine | Many, without coordination | Many, coordinated through the catalog |
| Typical weakness | Cost, lock-in, ML access | Consistency, governance | Operational complexity: catalogs, compaction, snapshot expiry |

:::{.fragment .callout-caution}
A lakehouse does not automatically replace an OLTP database or a low-latency serving store. Point lookups and high-rate small updates still belong in systems designed for them.
:::

## Lakehouse architecture

```{mermaid}
%%| echo: false
flowchart LR
    subgraph ENG["Engines"]
        BI["BI / SQL"]
        ML["ML / notebooks"]
        ST["Streaming"]
    end
    CAT["Catalog"]
    TF["Table format<br/>metadata"]
    FF["Data files"]
    STO["Storage"]

    BI --> CAT
    ML --> CAT
    ST --> CAT
    CAT -->|"pointer"| TF
    TF -->|"lists"| FF
    FF --> STO
    ENG -.->|"read / write"| FF
```

| Layer | Role | Examples |
|---|---|---|
| Engines | Query and process tables | ClickHouse, Trino, DuckDB, Spark, Flink, Ray |
| Catalog | Table names, atomic commits, access control | Iceberg REST, Polaris, Unity, Nessie, Glue, Hive |
| Table format | Snapshots, schemas, manifests | Iceberg, Delta Lake, Hudi |
| Data files | Encoded records | Parquet, ORC, Avro |
| Storage | Durable bytes | S3, MinIO / RustFS, HDFS, GCS, ADLS |

The lakehouse is an architectural pattern, not a single product: each layer can be chosen independently.

## The pieces fit together

```text
Spark SQL
  | Iceberg catalog: table name and current metadata
  v
Iceberg metadata files  --->  snapshots / schemas / manifests
  |
  +---- references Parquet data files
                 |
                 v
          HDFS or S3-compatible storage (RustFS, MinIO)
```

The catalog helps Spark resolve a table and coordinate commits; Iceberg metadata identifies the files in a snapshot; storage makes those files available to the Spark driver and executors.

### Snapshots and time travel (Iceberg, Spark SQL)

Every commit creates a **snapshot**. Readers use one consistent snapshot, and older snapshots remain queryable until they are expired.

```sql
-- table history
SELECT committed_at, snapshot_id, operation FROM demo.course.nyc311.snapshots;

-- schema evolution: metadata-only change, no data files are rewritten
ALTER TABLE demo.course.nyc311 ADD COLUMN resolved BOOLEAN;

-- time travel by snapshot or timestamp
SELECT count(*) FROM demo.course.nyc311 VERSION AS OF 1234567890123456789;
SELECT count(*) FROM demo.course.nyc311 TIMESTAMP AS OF '2026-11-15 10:00:00';

-- undo a bad write
CALL demo.system.rollback_to_snapshot('course.nyc311', 1234567890123456789);

-- maintenance: old snapshots and their files are not kept forever
CALL demo.system.expire_snapshots(table => 'course.nyc311', older_than => TIMESTAMP '2026-11-01 00:00:00');
```

Delta Lake offers equivalent operations: `DESCRIBE HISTORY`, `VERSION AS OF`, `RESTORE`, and `VACUUM`.

## Catalogs: the lakehouse control plane

A catalog maps `namespace.table` to the table's **current metadata** and performs the **atomic commit**. It is increasingly also responsible for **access control**, credential vending, and governance.

| Catalog | Origin | Notes |
|---|---|---|
| Hive Metastore | Apache Hive | Long-established; widely supported |
| Iceberg REST catalog | Apache Iceberg (specification) | Open HTTP API implemented by many servers |
| Apache Polaris | Snowflake, donated 2024 | Implements the Iceberg REST API |
| Unity Catalog | Databricks, open-sourced 2024 | Delta and Iceberg, plus files, ML models, and permissions |
| Project Nessie | Dremio | Git-like branches and tags for tables |
| Lakekeeper, Apache Gravitino | Open source | REST catalog / metadata federation |
| AWS Glue, Google BigLake metastore | Cloud vendors | Managed catalogs integrated with cloud IAM |

:::{.fragment .callout-note}
In lesson 23, OpenMetadata adds a governance view (ownership, lineage, quality) on top of these technical catalogs; it does not replace the table catalog.
:::

## Lakehouse vendors and platforms

| Vendor / project | Lakehouse offering | Primary table format | Notes |
|---|---|---|---|
| **Databricks** | Data Intelligence Platform, Unity Catalog | Delta Lake | Popularized the term; acquired Tabular (founded by Iceberg's creators) in 2024; reads Iceberg via UniForm |
| **Snowflake** | Iceberg tables, Snowflake Open Catalog | Iceberg | Warehouse vendor that moved toward open tables; created Polaris |
| **AWS** | S3 Tables, Glue Data Catalog, Athena, EMR, Lake Formation | Iceberg (also Hudi, Delta) | S3 Tables (2024) manages Iceberg tables in the storage service |
| **Google Cloud** | BigLake, BigQuery tables for Apache Iceberg | Iceberg | Governs open tables from BigQuery |
| **Microsoft** | Fabric OneLake | Delta Lake | One logical lake for all Fabric workloads |
| **Cloudera** | Open Data Lakehouse | Iceberg | Hybrid and on-premises deployments |
| **Dremio**, **Starburst** | SQL lakehouse engines (Dremio; Trino) | Iceberg (multi-format) | Query engines over open tables |
| **Onehouse** | Managed ingestion and tables | Hudi | Founded by Hudi's creator |
| **Open-source build (this course)** | Spark + Iceberg + REST catalog + S3-compatible storage + ClickHouse/DuckDB | Iceberg | No vendor lock-in; you operate every component |

:::{.fragment .callout-caution}
This landscape changes quickly, through acquisitions, renamed products, and new catalogs. Treat this table as a snapshot from November 2026 and verify the details before making a choice.
:::

## The lakehouse in the course project

```text
Telegram / pizza events -> Kafka (+ Schema Registry)
        -> Spark Structured Streaming (enrichment, lesson 15 AI inference)
        -> Iceberg tables on S3-compatible storage   <- historical, replayable, versioned
             |-> ClickHouse / Metabase (lesson 17)
             |-> Spark batch jobs, compaction, backfills (lesson 20)
             '-> OpenMetadata lineage and quality tests (lesson 23)
```

- **Kafka** is the retained event log; the **lakehouse** is the long-term analytical history.
- Time travel helps answer "what did the data look like when this dashboard or answer was produced?"
- Retention matters: delete or expire raw personal data according to the project's privacy choices, including old snapshots.

## Different persistence roles: Kafka, Elasticsearch, and a Lakehouse

| System | Typical role | What it is good at |
|---|---|---|
| **Kafka** | Retained event log | Decoupling producers and consumers; replaying events within the configured retention window |
| **Elasticsearch** (later lesson) | Search/serving index | Low-latency text, filter, and aggregation queries over indexed documents |
| **Iceberg on HDFS/object storage** | Analytical table and historical store | SQL analytics over large datasets, table snapshots, and long-term retention policies |

These are common roles, not hard limits: Kafka and Elasticsearch can retain data for a long time, and lakehouse tables can be expired or deleted. **“Long-term” describes the intended retention and analytical role, not an automatic durability guarantee.** Configure retention, replication, backup, and recovery for each system.

## Demo: pizza orders into the lakehouse

```{mermaid}
%%| echo: false
flowchart LR
    D["Datagen<br/>pizza_orders"] --> C["Kafka Connect<br/>Avro"]
    C --> K["Kafka<br/>2 brokers"]
    C -.-> R["Schema Registry"]
    K --> S["Spark Structured Streaming<br/>2 workers"]
    S -.-> R
    S -->|"commit snapshot"| CAT["Iceberg REST catalog"]
    S -->|"Parquet data + metadata"| M["RustFS<br/>s3://lakehouse"]
    CAT -.->|"current metadata pointer"| M
```

Run from `tap2027/quickstart/pizza-lakehouse` (free local ports: 14040, 18080, 18081, 18083, 18181, 19000, 19001):

```bash
docker compose up -d --build --wait broker1 broker2 registry connect rustfs iceberg-rest

docker compose exec broker1 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server broker1:9092 --create --if-not-exists \
  --topic pizza_orders --partitions 2 --replication-factor 2

curl --fail-with-body -X PUT -H 'Content-Type: application/json' \
  --data @connector.json http://localhost:18083/connectors/pizza-orders/config

docker compose --profile spark up -d spark
docker compose logs -f spark      # wait for "Created Iceberg table lake.pizza.orders"
```

Same Kafka setup as the Parquet demo; the differences are **no shared Docker volume for the workers** (they write to RustFS through the S3 API) and an **atomic commit per micro-batch** through the REST catalog.

### Demo: inspect storage, snapshots, and time travel

```bash
# objects in RustFS: Parquet data files + Iceberg metadata
# (or open http://localhost:19001/rustfs/console/index.html)
docker compose exec rustfs sh -c 'rc alias set local http://localhost:9000 tapadmin tapsecret123 >/dev/null &&
  rc object list --recursive local/lakehouse/warehouse/pizza/orders'

# SQL on the Iceberg table (the "lake" catalog is set in spark-defaults.conf)
docker compose exec spark /opt/spark/bin/spark-sql -S --conf spark.ui.enabled=false 2>/dev/null
```

```sql
SELECT count(*) FROM lake.pizza.orders;
SELECT committed_at, snapshot_id, operation, summary['added-records']
  FROM lake.pizza.orders.snapshots ORDER BY committed_at;
SELECT partition, record_count, file_count FROM lake.pizza.orders.partitions;
SELECT count(*) FROM lake.pizza.orders VERSION AS OF <snapshot_id>;   -- time travel
```

- Restart Spark (`docker compose stop spark` then `docker compose --profile spark up -d spark`): the checkpoint resumes the Kafka offsets and the catalog keeps the table, so no rows are duplicated.
- One snapshot every 10 seconds means many small files: discuss compaction (`CALL lake.system.rewrite_data_files('pizza.orders')`) and snapshot expiry.
- Stop with `docker compose --profile spark down`; add `-v` **only** to delete Kafka, RustFS, catalog, and checkpoint data.